# SA-CUT Training Pipeline — Google Colab

**SA-CUT**: Structure-Anchored Contrastive Unpaired Translation  
Virtual H&E staining from TPAF microscopy images.

## Prerequisites
- Runtime: GPU (A100 recommended)
- All data and the Cellpose checkpoint live on **Google Drive**
- SA-CUT repo cloned to `/content/SA-CUT`

## Steps
1. Fill in paths in **Cell 1** — that is the only cell you need to edit
2. Run all cells in order (Runtime → Run all)

---

## Cell 1 — ⚙️ User Configuration
**Edit the paths below, then run all cells.**

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# FILL IN YOUR PATHS HERE — this is the only cell you need to edit
# ═══════════════════════════════════════════════════════════════════════════

GDRIVE_ROOT = "/content/drive/MyDrive"

# Path to your fine-tuned Cellpose-SAM checkpoint file
# Example: "/content/drive/MyDrive/models/cpsam_20260228_gray"
CHECKPOINT_PATH = f"{GDRIVE_ROOT}/..."  # ← FILL IN

# Folder containing your 2557 TPAF PNG images (512×512, grayscale 8-bit)
TPAF_PNG_DIR = f"{GDRIVE_ROOT}/..."     # ← FILL IN

# Folder containing your 2532 H&E PNG images (512×512, RGB 24-bit)
HE_PNG_DIR = f"{GDRIVE_ROOT}/..."       # ← FILL IN

# ───────────────────────────────────────────────────────────────────────────
# Output directories on Google Drive (written here to survive session restarts)
# ───────────────────────────────────────────────────────────────────────────
TPAF_NPY_DIR = f"{GDRIVE_ROOT}/sa_cut_data/tpaf_npy"
HE_NPY_DIR   = f"{GDRIVE_ROOT}/sa_cut_data/he_npy"
MASK_NPY_DIR = f"{GDRIVE_ROOT}/sa_cut_data/masks_npy"

# ───────────────────────────────────────────────────────────────────────────
# Cellpose inference parameters
# diameter: expected nucleus diameter in pixels at 512×512 resolution
#   — 30 px is a safe default; increase if nuclei look under-segmented
# ───────────────────────────────────────────────────────────────────────────
CELLPOSE_MODEL_TYPE    = "cyto3"   # base model used when fine-tuning
CELLPOSE_DIAMETER      = 30        # px
CELLPOSE_FLOW_THRESH   = 0.4
CELLPOSE_CELLPROB_THRESH = 0.0

# ═══════════════════════════════════════════════════════════════════════════
print("Configuration set. Proceed to Cell 2.")

## Cell 2 — Mount Google Drive & Set Up Repository

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, sys

# Clone SA-CUT repo if not already present
if not os.path.exists("/content/SA-CUT"):
    !git clone https://github.com/YOUR_USERNAME/SA-CUT.git /content/SA-CUT  # ← update URL if needed

%cd /content/SA-CUT
sys.path.insert(0, "/content/SA-CUT")

# Create output directories on Google Drive
for d in [TPAF_NPY_DIR, HE_NPY_DIR, MASK_NPY_DIR]:
    os.makedirs(d, exist_ok=True)
    print(f"Ready: {d}")

## Cell 3 — Install Dependencies

In [ ]:
# Colab already has torch/torchvision; install the rest
!pip install -q cellpose tifffile tqdm pyyaml monai pytorch-fid

# Verify GPU
import torch
assert torch.cuda.is_available(), "No GPU detected — change runtime to GPU (Runtime → Change runtime type)"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"PyTorch: {torch.__version__}")

## Cell 4 — Verify Cellpose Model Loads

This cell gates everything else.  
Expected output: a side-by-side image showing TPAF and nucleus mask overlay.

In [ ]:
import numpy as np
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch

# ── 1. Load model ──────────────────────────────────────────────────────────
print(f"Loading Cellpose-SAM checkpoint from:\n  {CHECKPOINT_PATH}")
print(f"File size: {Path(CHECKPOINT_PATH).stat().st_size / 1e6:.1f} MB")

try:
    from cellpose import models as _cp_models
    cp_model = _cp_models.CellposeModel(
        pretrained_model=str(CHECKPOINT_PATH),
        model_type=CELLPOSE_MODEL_TYPE,
        gpu=True,
    )
    print("✓ Model loaded successfully")
except Exception as e:
    print(f"✗ Model loading FAILED: {e}")
    print("Possible fixes:")
    print("  • Check CHECKPOINT_PATH is correct")
    print("  • Try CELLPOSE_MODEL_TYPE = 'nuclei' or 'cyto2' if 'cyto3' fails")
    print("  • Ensure cellpose>=2.3 is installed (pip install -U cellpose)")
    raise

# ── 2. Pick a sample TPAF image ────────────────────────────────────────────
tpaf_pngs = sorted(Path(TPAF_PNG_DIR).glob("*.png"))
assert len(tpaf_pngs) > 0, f"No PNG files found in {TPAF_PNG_DIR}"
sample_path = tpaf_pngs[0]
print(f"\nRunning inference on: {sample_path.name}")

# ── 3. Load & normalise TPAF ───────────────────────────────────────────────
img_pil = Image.open(sample_path).convert("L")
img_np = np.array(img_pil, dtype=np.float32)       # (H, W) uint8 → float32
lo, hi = np.percentile(img_np, [1, 99])
img_norm = np.clip((img_np - lo) / (hi - lo + 1e-6), 0.0, 1.0)  # [0, 1]
img_cp   = (img_norm * 255.0).astype(np.float32)   # Cellpose expects [0, 255]

print(f"Image shape: {img_np.shape}, percentile range: [{lo:.1f}, {hi:.1f}]")

# ── 4. Run Cellpose inference ──────────────────────────────────────────────
with torch.no_grad():
    masks_inst, flows, _ = cp_model.eval(
        img_cp,
        diameter=CELLPOSE_DIAMETER,
        channels=[0, 0],              # grayscale
        flow_threshold=CELLPOSE_FLOW_THRESH,
        cellprob_threshold=CELLPOSE_CELLPROB_THRESH,
        normalize=True,
    )

binary_mask = (masks_inst > 0).astype(np.float32)  # (H, W)

# ── 5. Print stats ─────────────────────────────────────────────────────────
n_nuclei = int(masks_inst.max())
coverage = float(binary_mask.mean())
print(f"\nDetected nuclei : {n_nuclei}")
print(f"Coverage ratio  : {coverage:.3f}  ({coverage*100:.1f}% of pixels)")
if coverage < 0.02:
    print("⚠ Very low coverage — try decreasing CELLPOSE_DIAMETER or CELLPOSE_CELLPROB_THRESH")
elif coverage > 0.60:
    print("⚠ Very high coverage — try increasing CELLPOSE_CELLPROB_THRESH")
else:
    print("✓ Coverage looks reasonable")

# ── 6. Visualise ───────────────────────────────────────────────────────────
overlay = np.stack([img_norm, img_norm, img_norm], axis=-1)  # (H, W, 3) gray
overlay[binary_mask > 0] = [1.0, 0.2, 0.2]                  # red = nucleus

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img_norm, cmap="gray");       axes[0].set_title("TPAF (normalised)")
axes[1].imshow(binary_mask, cmap="gray");    axes[1].set_title(f"Binary Mask ({n_nuclei} nuclei)")
axes[2].imshow(overlay);                     axes[2].set_title(f"Overlay (coverage={coverage:.2f})")
for ax in axes: ax.axis("off")
plt.tight_layout()
plt.savefig("/content/cellpose_verify.png", dpi=120, bbox_inches="tight")
plt.show()
print("\n✓ Verification complete. If the mask looks correct, proceed to Cell 5.")

## Cell 5 — Generate Masks for All TPAF Images

Uses the existing `scripts/precompute_masks.py`.  
Outputs: one `.npy` mask per TPAF image in `MASK_NPY_DIR`, plus `qc_summary.csv` and `qc_montage.png`.

**Runtime:** ~0.5–2 s per image on A100 → ~20–85 min for 2557 images.

In [ ]:
# Check how many masks already exist (safe to re-run; existing files are skipped)
from pathlib import Path
existing_masks = list(Path(MASK_NPY_DIR).glob("*.npy"))
print(f"Existing masks in output dir: {len(existing_masks)}")

!python /content/SA-CUT/scripts/precompute_masks.py \
    --input_dir  "{TPAF_PNG_DIR}" \
    --output_dir "{MASK_NPY_DIR}" \
    --checkpoint "{CHECKPOINT_PATH}" \
    --model_type  {CELLPOSE_MODEL_TYPE} \
    --diameter    {CELLPOSE_DIAMETER} \
    --flow_threshold   {CELLPOSE_FLOW_THRESH} \
    --cellprob_threshold {CELLPOSE_CELLPROB_THRESH}

In [ ]:
# Display QC results
import pandas as pd
from pathlib import Path
from PIL import Image as PILImage
import matplotlib.pyplot as plt
import IPython.display as ipd

qc_csv = Path(MASK_NPY_DIR) / "qc_summary.csv"
if qc_csv.exists():
    df = pd.read_csv(qc_csv)
    print(df.describe().round(4))
    low_cov = df[df["coverage_ratio"] < 0.02]
    if len(low_cov):
        print(f"\n⚠ {len(low_cov)} patches have coverage < 2% — consider reviewing them")
        print(low_cov[["filename", "coverage_ratio", "num_nuclei"]].head(10).to_string())

qc_montage = Path(MASK_NPY_DIR) / "qc_montage.png"
if qc_montage.exists():
    img = PILImage.open(qc_montage)
    plt.figure(figsize=(16, 16))
    plt.imshow(img); plt.axis("off"); plt.title("QC Montage (sample TPAF patches + masks)")
    plt.tight_layout(); plt.show()

total_masks = len(list(Path(MASK_NPY_DIR).glob("*.npy")))
print(f"\nTotal masks generated: {total_masks}")

## Cell 6 — Convert PNG Patches to NPY

The dataset loader (`data/dataset.py`) only reads `.npy` files.  
This cell converts PNGs to float32 NPY and writes them to Google Drive.

**TPAF:** percentile-clipped [1%–99%] → float32 [0, 1], shape `(H, W)`  
**H&E:** uint8 / 255 → float32 [0, 1], shape `(H, W, 3)`

**Runtime:** ~5–15 min for ~5000 images.

In [ ]:
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm.notebook import tqdm


def convert_tpaf_png_to_npy(src_dir: str, dst_dir: str) -> int:
    """Convert TPAF grayscale PNGs to float32 NPY with percentile normalisation."""
    src = Path(src_dir)
    dst = Path(dst_dir)
    dst.mkdir(parents=True, exist_ok=True)
    pngs = sorted(src.glob("*.png"))
    skipped = 0
    for p in tqdm(pngs, desc="TPAF PNG→NPY"):
        out = dst / (p.stem + ".npy")
        if out.exists():   # resume-safe
            skipped += 1
            continue
        img = np.array(Image.open(p).convert("L"), dtype=np.float32)
        lo, hi = np.percentile(img, [1.0, 99.0])
        img = np.clip((img - lo) / (hi - lo + 1e-6), 0.0, 1.0)
        np.save(str(out), img)          # shape: (H, W)
    print(f"  Skipped (already done): {skipped} / {len(pngs)}")
    return len(pngs)


def convert_he_png_to_npy(src_dir: str, dst_dir: str) -> int:
    """Convert H&E RGB PNGs to float32 NPY, scaled to [0, 1]."""
    src = Path(src_dir)
    dst = Path(dst_dir)
    dst.mkdir(parents=True, exist_ok=True)
    pngs = sorted(src.glob("*.png"))
    skipped = 0
    for p in tqdm(pngs, desc="H&E  PNG→NPY"):
        out = dst / (p.stem + ".npy")
        if out.exists():
            skipped += 1
            continue
        img = np.array(Image.open(p).convert("RGB"), dtype=np.float32) / 255.0
        np.save(str(out), img)          # shape: (H, W, 3)
    print(f"  Skipped (already done): {skipped} / {len(pngs)}")
    return len(pngs)


n_tpaf = convert_tpaf_png_to_npy(TPAF_PNG_DIR, TPAF_NPY_DIR)
n_he   = convert_he_png_to_npy(HE_PNG_DIR,   HE_NPY_DIR)

print(f"\nTPAF: {n_tpaf} images → {TPAF_NPY_DIR}")
print(f"H&E : {n_he}   images → {HE_NPY_DIR}")

# ── Quick sanity check ──────────────────────────────────────────────────────
sample_tpaf = next(Path(TPAF_NPY_DIR).glob("*.npy"))
sample_he   = next(Path(HE_NPY_DIR).glob("*.npy"))
sample_mask = next(Path(MASK_NPY_DIR).glob("*.npy"))

t = np.load(str(sample_tpaf))
h = np.load(str(sample_he))
m = np.load(str(sample_mask))

print(f"\n── Sanity check ─────────────────────────────")
print(f"TPAF  | shape={t.shape} dtype={t.dtype} range=[{t.min():.3f}, {t.max():.3f}]")
print(f"H&E   | shape={h.shape} dtype={h.dtype} range=[{h.min():.3f}, {h.max():.3f}]")
print(f"Mask  | shape={m.shape} dtype={m.dtype} range=[{m.min():.3f}, {m.max():.3f}]")

assert t.shape == (512, 512),      f"Expected TPAF shape (512, 512), got {t.shape}"
assert h.shape == (512, 512, 3),   f"Expected H&E shape (512, 512, 3), got {h.shape}"
assert m.dtype == np.float32,      f"Mask dtype should be float32, got {m.dtype}"
assert t.dtype == np.float32,      f"TPAF dtype should be float32, got {t.dtype}"
assert 0.0 <= t.max() <= 1.0,     f"TPAF values out of [0,1]: max={t.max()}"
print("\n✓ All assertions passed")

## Cell 7 — Write Experiment Configuration

Writes `configs/experiment_real_data.yaml` with the correct paths and `patch_size: 512`.

In [ ]:
import yaml
from pathlib import Path
from datetime import date

today = date.today().strftime("%Y%m%d")

cfg = {
    # Inherits all unspecified keys from configs/default.yaml at runtime
    "experiment": {
        "name": f"sa_cut_{today}_real_data",
        "use_wandb": False,
        "use_tensorboard": True,
    },
    "data": {
        "tpaf_dir": str(TPAF_NPY_DIR),
        "he_dir":   str(HE_NPY_DIR),
        "patch_size": 512,         # actual image size
        "tpaf_channels": 1,
        "num_workers": 2,          # Colab recommends ≤2 workers
    },
    "mask_provider": {
        "mode": "precomputed",
        "mask_dir": str(MASK_NPY_DIR),
        # checkpoint documented for reference; not used in precomputed mode
        "cellpose_checkpoint": str(CHECKPOINT_PATH),
        "cellpose_model_type": CELLPOSE_MODEL_TYPE,
        "cellpose_diameter": CELLPOSE_DIAMETER,
        "log_mask_stats": True,
    },
    "generator": {
        "input_nc": 2,   # TPAF (1ch) + Mask (1ch)
        "output_nc": 3,
        "ngf": 64,
        "n_resnet_blocks": 9,
        "norm_type": "instance",
        "use_dropout": False,
        "mask_injection": "early_fusion",
    },
    "discriminator": {
        "input_nc": 3,
        "ndf": 64,
        "n_layers": 3,
        "norm_type": "instance",
    },
    "losses": {
        "lambda_adv": 1.0,
        "lambda_patchnce": 1.0,
        "lambda_struct": 5.0,
        "lambda_idt": 0.5,
        "gan_mode": "lsgan",
        "n_patchnce_layers": 5,
        "n_patches_per_layer": 256,
        "struct_loss_mode": "threshold",
        "struct_warmup_epochs": 10,
        "struct_rampup_epochs": 5,
    },
    "training": {
        "batch_size": 1,
        "lr_G": 2.0e-4,
        "lr_D": 2.0e-4,
        "beta1": 0.5,
        "beta2": 0.999,
        "n_epochs": 200,
        "n_epochs_decay": 200,
        "seed": 42,
        "use_amp": True,   # Mixed precision — recommended on A100 for 512×512
    },
}

config_path = Path("/content/SA-CUT/configs/experiment_real_data.yaml")
with open(config_path, "w") as f:
    yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)

print(f"Config written to: {config_path}")
print("\n── Contents ───────────────────────────────────")
print(config_path.read_text())

## Cell 8 — Training

**400 total epochs** (200 full LR + 200 linear decay).  
Checkpoints saved every 10 epochs to `checkpoints/`.  
TensorBoard logs written to `results/logs/`.

To resume from a checkpoint, uncomment the `--resume` line.

In [ ]:
# Launch TensorBoard (optional — open the link in a new tab)
%load_ext tensorboard
%tensorboard --logdir /content/SA-CUT/results/logs

In [ ]:
!python /content/SA-CUT/scripts/train.py \
    --config /content/SA-CUT/configs/experiment_real_data.yaml
    # --resume /content/SA-CUT/checkpoints/sa_cut_XXXXXX_real_data/latest.pth

## Cell 9 — Save Checkpoints to Google Drive

Run this after training (or periodically) to back up checkpoints to Drive.  
Colab local storage is wiped when the session ends.

In [ ]:
import shutil
from pathlib import Path

CKPT_BACKUP_DIR = f"{GDRIVE_ROOT}/sa_cut_checkpoints"

src = Path("/content/SA-CUT/checkpoints")
dst = Path(CKPT_BACKUP_DIR)
dst.mkdir(parents=True, exist_ok=True)

for ckpt_dir in src.iterdir():
    if ckpt_dir.is_dir():
        dest_dir = dst / ckpt_dir.name
        shutil.copytree(str(ckpt_dir), str(dest_dir), dirs_exist_ok=True)
        print(f"Backed up: {ckpt_dir.name}")

print(f"\nAll checkpoints backed up to:\n  {CKPT_BACKUP_DIR}")